### Chosen Task
For this assignment, I selected **POS Tagging** using the chosen dataset and implemented token classification using BERT. Chunking is discussed in the comparison section for conceptual understanding.

In [2]:
from datasets import load_dataset

dataset = load_dataset("a3lem/universal-dependencies-parquet", "en_ewt")
dataset

README.md: 0.00B [00:00, ?B/s]

data/en_ewt/test-00000-of-00001.parquet:   0%|          | 0.00/519k [00:00<?, ?B/s]

data/en_ewt/validation-00000-of-00001.pa(…):   0%|          | 0.00/516k [00:00<?, ?B/s]

data/en_ewt/train-00000-of-00001.parquet:   0%|          | 0.00/3.61M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/2077 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2001 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/12544 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['sent_id', 'text', 'ids', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'heads', 'deprels', 'deps', 'misc'],
        num_rows: 2077
    })
    validation: Dataset({
        features: ['sent_id', 'text', 'ids', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'heads', 'deprels', 'deps', 'misc'],
        num_rows: 2001
    })
    train: Dataset({
        features: ['sent_id', 'text', 'ids', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'heads', 'deprels', 'deps', 'misc'],
        num_rows: 12544
    })
})

In [4]:
dataset["train"][0]

{'sent_id': 'weblog-juancole.com_juancole_20051126063000_ENG_20051126_063000-0001',
 'text': 'Al-Zaman : American forces killed Shaikh Abdullah al-Ani, the preacher at the mosque in the town of Qaim, near the Syrian border.',
 'ids': ['1',
  '2',
  '3',
  '4',
  '5',
  '6',
  '7',
  '8',
  '9',
  '10',
  '11',
  '12',
  '13',
  '14',
  '15',
  '16',
  '17',
  '18',
  '19',
  '20',
  '21',
  '22',
  '23',
  '24',
  '25',
  '26',
  '27',
  '28',
  '29'],
 'tokens': ['Al',
  '-',
  'Zaman',
  ':',
  'American',
  'forces',
  'killed',
  'Shaikh',
  'Abdullah',
  'al',
  '-',
  'Ani',
  ',',
  'the',
  'preacher',
  'at',
  'the',
  'mosque',
  'in',
  'the',
  'town',
  'of',
  'Qaim',
  ',',
  'near',
  'the',
  'Syrian',
  'border',
  '.'],
 'lemmas': ['Al',
  '-',
  'Zaman',
  ':',
  'American',
  'force',
  'kill',
  'Shaikh',
  'Abdullah',
  'al',
  '-',
  'Ani',
  ',',
  'the',
  'preacher',
  'at',
  'the',
  'mosque',
  'in',
  'the',
  'town',
  'of',
  'Qaim',
  ',',
  'near',
 

In [6]:
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)

In [7]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(examples["upos"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [11]:
all_labels = set()

for example in dataset["train"]:
    for tag in example["upos"]:
        all_labels.add(tag)

label_list = sorted(list(all_labels))

label_to_id = {label: i for i, label in enumerate(label_list)}
id_to_label = {i: label for label, i in label_to_id.items()}

print(label_list)
print("Number of labels:", len(label_list))

['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X', '_']
Number of labels: 18


In [12]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["upos"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label_to_id[label[word_idx]])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [13]:
tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/2077 [00:00<?, ? examples/s]

Map:   0%|          | 0/2001 [00:00<?, ? examples/s]

Map:   0%|          | 0/12544 [00:00<?, ? examples/s]

In [14]:
num_labels = len(label_list)

model = AutoModelForTokenClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels,
    id2label=id_to_label,
    label2id=label_to_id
)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized be

In [15]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [17]:
import evaluate

In [18]:
seqeval = evaluate.load("seqeval")

In [19]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for pred, lab in zip(predictions, labels):
        current_preds = []
        current_labels = []

        for p_val, l_val in zip(pred, lab):
            if l_val != -100:
                current_preds.append(id_to_label[p_val])
                current_labels.append(id_to_label[l_val])

        true_predictions.append(current_preds)
        true_labels.append(current_labels)

    results = seqeval.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [20]:
training_args = TrainingArguments(
    output_dir="./pos_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    logging_dir="./logs",
    load_best_model_at_end=True,
    fp16=True
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [22]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [24]:
small_train = tokenized_datasets["train"].shuffle(seed=42).select(range(2000))
small_val = tokenized_datasets["validation"].shuffle(seed=42).select(range(500))

In [25]:
print(small_train)
print(small_val)

Dataset({
    features: ['sent_id', 'text', 'ids', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'heads', 'deprels', 'deps', 'misc', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 2000
})
Dataset({
    features: ['sent_id', 'text', 'ids', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'heads', 'deprels', 'deps', 'misc', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 500
})


In [26]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [28]:
import numpy as np

In [29]:
trainer.evaluate()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.282404,0.912823,0.906240,0.909519,0.924185


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ADJ seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NOUN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: ADP seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PUNCT seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: SCONJ seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171

{'eval_loss': 0.2824037969112396,
 'eval_precision': 0.9128231120121643,
 'eval_recall': 0.906239516940624,
 'eval_f1': 0.9095194007238449,
 'eval_accuracy': 0.9241854636591479}

In [30]:
sentence = "John works at Google in California"
words = sentence.split()

inputs = tokenizer(words, is_split_into_words=True, return_tensors="pt", truncation=True)

outputs = model(**inputs)
predictions = outputs.logits.argmax(dim=2).squeeze().tolist()

word_ids = inputs.word_ids()
predicted_labels = []

previous_word_id = None
for pred, word_id in zip(predictions, word_ids):
    if word_id is not None and word_id != previous_word_id:
        predicted_labels.append(id_to_label[pred])
    previous_word_id = word_id

for word, label in zip(words, predicted_labels):
    print(f"{word}: {label}")

John: PROPN
works: VERB
at: ADP
Google: PROPN
in: ADP
California: PROPN


Comparison: POS Tagging vs Chunking

Part-of-Speech (POS) tagging and Chunking are both important sequence-labeling tasks in Natural Language Processing, but they operate at different levels of linguistic understanding.

POS tagging assigns grammatical labels to individual words such as nouns, verbs, adjectives, and prepositions. It focuses on identifying the syntactic role of each word in a sentence. For example:

Sentence: John works at Google in California

POS tagging identifies:
John → Proper Noun
works → Verb
Google → Proper Noun
California → Proper Noun

Chunking, on the other hand, groups words into meaningful phrases such as noun phrases (NP) and verb phrases (VP). Instead of labeling single words, chunking identifies relationships between neighboring words and forms higher-level structures.

Example:
John → B-NP
works → B-VP
at Google → B-PP + B-NP
in California → B-PP + B-NP

POS tagging is generally considered easier because it focuses on word-level classification, while chunking is slightly more complex since it requires phrase-level grouping and contextual understanding between words.

Overall, POS tagging helps in grammar-level understanding, whereas chunking helps identify meaningful phrase structures within sentences.

Observations and Challenges Faced

During this assignment, I implemented token classification using a transformer-based model (BERT) for Part-of-Speech (POS) tagging. One of the key challenges was aligning word-level labels with subword tokens generated by the BERT tokenizer. This required careful handling of special tokens and assigning -100 to ignored positions so they would not affect training.

Another important observation was that transformer-based models like BERT significantly improve sequence labeling performance compared to traditional machine learning approaches because they capture contextual relationships between words more effectively.

I also learned how POS tagging operates at the grammatical level, while chunking works at the phrase level, making chunking slightly more complex than POS tagging. Overall, this assignment helped me understand token classification workflows including tokenization, label alignment, model training, evaluation using sequence metrics, and performing inference on custom sentences.